In [1]:
%autosave 60
%pip install --quiet -r requirements.txt

Autosaving every 60 seconds
Note: you may need to restart the kernel to use updated packages.


## Setup

In [ ]:
import os
import numpy as np
import random
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
from collections import defaultdict
from utils.hunt_data_loader import HuntDataLoader
from utils.train_eval import fit_3D, fit_3D_gan
from utils.loss_functions import recon_loss, ssim_L1_2d_loss
from models.alzheiminator_3d import ResidualUNet3D, Discriminator3D, Generator3D
from models.no_more_alzheimer_2d import UNet2D

data_loader = HuntDataLoader()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Load models

We load our best peforming 2D model

In [ ]:
output_dir = "out/unet_models"
model_path = f"{output_dir}/2d_unet_model.pt"
best_2d = UNet2D(latent_dim=64, in_channels=1, out_channels=1, base_ch=32).to(device)
best_2d.load_state_dict(torch.load(model_path))

We load our best performing 3D model

In [ ]:
output_dir = "out/unet_models"
model_path = f"{output_dir}/3d_unet_model.pt"
best_3d = ResidualUNet3D(in_ch=1, base=32).to(device)
best_3d.load_state_dict(torch.load(model_path))

## Display evenly spaced results from both models

In [ ]:
test_images = 4

# Same split as during training
training_pairs, test_pairs = data_loader.split_training_test_paths(seed=69)
print("We have", len(training_pairs), "training pairs")
print("We have", len(test_pairs), "test pairs")

In [ ]:
for i in range(test_images):
    padding = 15
    slice = int(padding + i * ((192 - 2 * padding) / test_images))
    candidate = random.randint(0, len(test_pairs)-1)

    # Generate 2D prediction
    x_num, y_num = data_loader.get_slice(data_path=test_pairs[candidate][0], crop_size=(192, 224), index=slice), data_loader.get_slice(data_path=test_pairs[candidate][1], crop_size=(192, 224), index=slice)
    x = data_loader.to_torch_img(x_num, device)
    y = data_loader.to_torch_img(y_num, device)
    recon_tensor_2d, _, _ = best_2d(x)

    # Generate 3D prediction
    x_num, y_num = data_loader.load_from_path(data_path=test_pairs[candidate][0], crop_size=(64, 192, 224)), data_loader.load_from_path(data_path=test_pairs[candidate][1], crop_size=(64, 192, 224))
    x = data_loader.to_torch_img(x_num, device)
    y = data_loader.to_torch_img(y_num, device)
    recon_tensors_3d = best_3d(x)

    # Convert Tensor reconstructions to numpy for display
    recon_2d = data_loader.to_numpy_img(recon_tensor_2d)
    recon_3d = data_loader.to_numpy_img(recon_tensors_3d)

    # How much the models change from the input
    change_2d = np.abs(recon_2d - x_num)
    change_3d = np.abs(recon_3d - x_num)

    # Difference between reconstructions and ground truth
    diff_2d = np.abs(recon_2d - y_num)
    diff_3d = np.abs(recon_3d - y_num)
    hunt_diff = np.abs(x_num - y_num)

    # Prediction errors
    pred_error_2d = ssim_L1_2d_loss(recon_tensor_2d, y).item()
    pred_error_3d = ssim_L1_2d_loss(recon_tensors_3d[slice], y).item()
    print(f"Displaying results for test image {candidate}, slice {slice}")

    # Hunt 3, Hunt 4, and the difference between them
    data_loader.display_slices(
        slices=[x_num, y_num, hunt_diff],
        slice_labels=["HUNT 3", "HUNT 4", "HUNT 3 vs HUNT 4 Diff"],
        slice_colors=["gray", "gray", "hot"]
    )

    # The four models’ outputs
    data_loader.display_slices(
        slices=[recon_2d, recon_3d],
        slice_labels=["2D Recon", "3D Recon"],
        slice_colors=["gray", "gray"]
    )

    # The four models’ change from HUNT 3
    data_loader.display_slices(
        slices=[change_2d, change_3d],
        slice_labels=["2D Change", "3D Change"],
        slice_colors=["hot", "hot"]
    )

    # The four models’ difference from HUNT 4
    data_loader.display_slices(
        slices=[diff_2d, diff_3d],
        slice_labels=[
            f"Slice Loss ({pred_error_2d:.3f})",
            f"Slice loss ({pred_error_3d:.3f})",
        ],
        slice_colors=["hot", "hot"]
    )

## Compare loss over entire volume for both volumes

As the 2D model only generates slices, it has to be run individually over all slices in a volume